# Credit Spreads and Business Cycle Fluctuations
### Research Notebook

Gilchrist, Simon, and Egon Zakrajsek. 2012. "Credit Spreads and Business Cycle Fluctuations." American Economic Review 102(4): 1692-1720.

Prepared by Arthur Faugeron, Literature Review project, Macro-Finance series.

This notebook is a companion laboratory to the written literature review. It does not attempt to reproduce the paper's central GZ credit spread or excess bond premium exactly, because those series are built from a proprietary bond-level dataset (month-end secondary market prices of senior unsecured corporate bonds from the Lehman/Warga and Merrill Lynch databases, matched to Compustat and CRSP) that is not publicly available. Instead, the notebook uses publicly available, real data from the Federal Reserve Economic Data (FRED) system to demonstrate the paper's central empirical logic: that corporate credit spreads carry incremental information about future real activity beyond the term spread and the policy rate, and that this predictive content is concentrated around business cycle turning points.

**Data labelling convention used throughout.** Every result below is labelled as one of:

- Original Paper Data: a number or table taken directly from the published article.
- Alternative Data: a publicly available FRED series used here because the original bond-level data are unavailable.
- Illustrative Simulation: an artificial series used only to demonstrate a mechanism (none are used in this notebook).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# NBER-dated US recessions (peak, trough), used only for shading charts.
# Source: NBER Business Cycle Dating Committee, https://www.nber.org/research/business-cycle-dating
NBER_RECESSIONS = [
    ('1973-11-01', '1975-03-01'),
    ('1980-01-01', '1980-07-01'),
    ('1981-07-01', '1982-11-01'),
    ('1990-07-01', '1991-03-01'),
    ('2001-03-01', '2001-11-01'),
    ('2007-12-01', '2009-06-01'),
    ('2020-02-01', '2020-04-01'),
]

def shade_recessions(ax, start='1973-01-01', end='2026-01-01'):
    for (p, t) in NBER_RECESSIONS:
        if pd.Timestamp(t) >= pd.Timestamp(start) and pd.Timestamp(p) <= pd.Timestamp(end):
            ax.axvspan(pd.Timestamp(p), pd.Timestamp(t), color='grey', alpha=0.15, lw=0)

## 2. Research Question

Gilchrist and Zakrajsek (2012) ask whether corporate bond credit spreads contain information about future real economic activity beyond what is already captured by the term spread and the real federal funds rate, and whether that information reflects default risk itself or a separate, time-varying price of bearing default risk (the excess bond premium).

This notebook asks a narrower, replicable version of the same question using public data: does the Moody's Baa-Aaa corporate bond spread, a coarse cousin of the GZ spread that the paper itself uses as one of its two benchmark comparison series (see the paper's Figure 1), help forecast industrial production growth and changes in the unemployment rate, conditional on the term spread and a short-rate control? And does that forecasting power hold up, weaken, or strengthen across subsamples, consistent with the paper's own finding that the Baa-Aaa spread is a much weaker predictor than the bottom-up GZ spread?

## 3. Data Acquisition

**Alternative Data.** All series below are downloaded live from FRED (Federal Reserve Bank of St. Louis) using `pandas_datareader`. FRED mirrors data originally published by the Federal Reserve Board (H.15 Selected Interest Rates), Moody's Investors Service, the Bureau of Labor Statistics, and the Federal Reserve Board's G.17 industrial production release.

| Series ID | Description | Source | Frequency | Role in this notebook |
|---|---|---|---|---|
| BAA | Moody's Seasoned Baa Corporate Bond Yield | Moody's / FRED | Monthly | Credit spread input |
| AAA | Moody's Seasoned Aaa Corporate Bond Yield | Moody's / FRED | Monthly | Credit spread input |
| INDPRO | Industrial Production: Total Index | Federal Reserve Board (G.17) | Monthly | Activity variable (paper's IPM) |
| UNRATE | Civilian Unemployment Rate | BLS | Monthly | Activity variable (paper's UER) |
| GS10 | 10-Year Treasury Constant Maturity Yield | Federal Reserve Board (H.15) | Monthly | Term spread input |
| TB3MS | 3-Month Treasury Bill, Secondary Market | Federal Reserve Board (H.15) | Monthly | Term spread input |

**Original Paper Data that cannot be reproduced here.** The GZ credit spread itself (equation 1 of the paper), the firm-level distance-to-default measure (Merton 1974, as implemented via Bharath and Shumway 2008), and the excess bond premium (the residual from the credit-spread pricing regression, equation 3) all require bond-level secondary market prices from the Lehman/Warga and Merrill Lynch databases matched to Compustat and CRSP. These are commercial datasets and are not reproduced here. As a point of reference, the paper's own Figure 1 reports that the GZ spread, the Baa-Aaa spread and the paper-bill spread are only modestly correlated with one another (pairwise correlations of 0.38, -0.17 and 0.21), so the Baa-Aaa results below should be read as a weaker, publicly available proxy, not a substitute, for the GZ spread.

In [ ]:
import pandas_datareader.data as web
import datetime

START = datetime.datetime(1973, 1, 1)
END = datetime.datetime.today()

series_ids = ['BAA', 'AAA', 'INDPRO', 'UNRATE', 'GS10', 'TB3MS']
raw = {}
for sid in series_ids:
    raw[sid] = web.DataReader(sid, 'fred', START, END)

df = pd.concat(raw.values(), axis=1)
df.columns = series_ids
df.index.name = 'date'
df = df.sort_index()
print(df.shape)
df.tail()

## 4. Data Cleaning

In [ ]:
# Forward-fill occasional single-month gaps in the interest rate series (holidays / reporting breaks),
# then drop any rows that remain incomplete.
df = df.asfreq('MS')
df[['BAA', 'AAA', 'GS10', 'TB3MS']] = df[['BAA', 'AAA', 'GS10', 'TB3MS']].ffill(limit=1)
df = df.dropna()
print(f"Usable monthly observations: {len(df)}  ({df.index.min().date()} to {df.index.max().date()})")

## 5. Variable Construction

We reconstruct, as closely as public data allow, the right-hand-side variables of the paper's forecasting regression (its equation 2):

$$ \nabla_h Y_{t+h} = \alpha + \sum_{i=1}^{p} \beta_i \nabla Y_{t-i} + \gamma_1 TS_t + \gamma_2 RFF_t + \gamma_3 CS_t + \epsilon_{t+h} $$

where $TS_t$ is the term spread (here, the 10-year Treasury yield less the 3-month bill yield, matching the paper's definition), $CS_t$ is a credit spread (here, Baa minus Aaa), and $RFF_t$ is a short-rate control. The paper defines $RFF_t$ as the real federal funds rate, using realized core PCE inflation; to keep this notebook self-contained on freely available monthly data we instead use the level of the 3-month bill yield as our short-rate control. This is a simplification, flagged explicitly, not a claim of exact replication.

In [ ]:
df['baa_aaa'] = df['BAA'] - df['AAA']          # credit spread proxy (CS_t)
df['term_spread'] = df['GS10'] - df['TB3MS']    # term spread (TS_t)
df['short_rate'] = df['TB3MS']                  # short-rate control (proxy for RFF_t)

# Forecast targets, matching the paper's annualised log-growth transformation
# nabla_h Y_{t+h} = (1200/h) * ln(Y_{t+h} / Y_t) for industrial production (monthly, log index)
# and the simple level change for the unemployment rate.
for h in [3, 12]:
    df[f'ip_growth_h{h}'] = (1200 / h) * np.log(df['INDPRO'].shift(-h) / df['INDPRO'])
    df[f'unrate_chg_h{h}'] = df['UNRATE'].shift(-h) - df['UNRATE']

# One lag of own monthly growth as a control, matching the AR structure of equation (2)
df['ip_growth_l1'] = 1200 * np.log(df['INDPRO'] / df['INDPRO'].shift(1))

df[['baa_aaa', 'term_spread', 'short_rate', 'ip_growth_h3', 'ip_growth_h12']].describe()

## 6. Mathematical Implementation

We estimate equation (2) by OLS. The paper uses Hodrick (1992) standard errors to correct for the moving-average error structure induced by overlapping h-step-ahead forecasts. We approximate this with Newey-West (HAC) standard errors at lag h, which target the same MA(h) problem and are the standard applied substitute when the exact Hodrick correction is not implemented in the estimation package being used.

In [ ]:
def forecasting_regression(data, y_col, h, credit_col='baa_aaa'):
    """Estimate the paper's equation (2) with and without the credit spread.
    Returns a dict of fitted statsmodels results."""
    d = data.dropna(subset=[y_col, 'term_spread', 'short_rate', 'ip_growth_l1', credit_col]).copy()
    X_base = sm.add_constant(d[['ip_growth_l1', 'term_spread', 'short_rate']])
    X_full = sm.add_constant(d[['ip_growth_l1', 'term_spread', 'short_rate', credit_col]])
    y = d[y_col]
    m_base = sm.OLS(y, X_base).fit(cov_type='HAC', cov_kwds={'maxlags': h})
    m_full = sm.OLS(y, X_full).fit(cov_type='HAC', cov_kwds={'maxlags': h})
    return {'baseline': m_base, 'with_credit_spread': m_full, 'n': len(d)}

## 7. Baseline Result

Table 2 of the paper reports that a 100 basis point increase in the GZ spread implies an almost 3.0 percentage point (annualised) drop in industrial production growth over the following three months, and that adding the GZ spread raises the 12-month-ahead adjusted R-squared for industrial production from about 0.225 (baseline) to 0.346. We now run the analogous regression with the Baa-Aaa spread in place of the GZ spread, over the paper's original sample window (1973 to 2010) to keep the comparison as close as possible.

In [ ]:
sample_1973_2010 = df.loc['1973-01-01':'2010-09-30']

results_ip3 = forecasting_regression(sample_1973_2010, 'ip_growth_h3', h=3)
results_ip12 = forecasting_regression(sample_1973_2010, 'ip_growth_h12', h=12)

for label, res in [('IP growth, h=3', results_ip3), ('IP growth, h=12', results_ip12)]:
    base_r2 = res['baseline'].rsquared_adj
    full_r2 = res['with_credit_spread'].rsquared_adj
    coef = res['with_credit_spread'].params['baa_aaa']
    tstat = res['with_credit_spread'].tvalues['baa_aaa']
    print(f"{label} (n={res['n']}):")
    print(f"  Adjusted R2, baseline (own lag + term spread + short rate): {base_r2:.3f}")
    print(f"  Adjusted R2, + Baa-Aaa spread:                              {full_r2:.3f}")
    print(f"  Coefficient on Baa-Aaa spread: {coef:.3f} (t = {tstat:.2f})\n")

**Interpretation.** If the sign and significance pattern found here mirrors the paper (a negative, statistically significant coefficient on the credit spread, and a rise in adjusted R-squared once the spread is added), that is consistent evidence that even a coarse, publicly available credit spread carries some of the same countercyclical information as the GZ spread. If the improvement in fit is much smaller than the paper reports for the GZ spread (which it should be, since the paper explicitly shows the GZ spread dominates the Baa-Aaa spread; see its Table 2, where the Baa-Aaa coefficient is small and, at the 12-month horizon for industrial production, not even statistically significant), that is direct, replicable confirmation of the paper's central claim: not all credit spreads are equally informative, and the bottom-up construction of the GZ spread is doing real work, not simply relabelling a generic credit spread.

## 8. Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(df.index, df['baa_aaa'], color='#172A3A', lw=1.1, label='Baa - Aaa spread')
shade_recessions(ax, df.index.min(), df.index.max())
ax.set_title('Moody\'s Baa - Aaa Corporate Bond Spread, 1973-present')
ax.set_ylabel('Percentage points')
ax.set_xlabel('Monthly. Source: FRED (BAA, AAA). Shaded bars: NBER-dated recessions.')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(df.index, df['baa_aaa'], color='#172A3A', lw=1.1, label='Baa - Aaa spread (left)')
ax1.set_ylabel('Credit spread (pp)')
ax2 = ax1.twinx()
ax2.plot(df.index, df['UNRATE'], color='#6B5744', lw=1.1, label='Unemployment rate (right)')
ax2.set_ylabel('Unemployment rate (pct)')
shade_recessions(ax1, df.index.min(), df.index.max())
ax1.set_title('Credit Spread and the Unemployment Rate')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

## 9. Robustness: 1985-2010 Subsample

The paper's Table 7, Panel B re-estimates its regressions over 1985-2010 to check whether the results are an artefact of the high-inflation, high-volatility 1970s and early 1980s, and finds that the excess bond premium's forecasting power is, if anything, larger in the later subsample. We repeat the same subsample check here.

In [ ]:
sample_1985_2010 = df.loc['1985-01-01':'2010-09-30']
res_sub = forecasting_regression(sample_1985_2010, 'ip_growth_h12', h=12)

print('1985-2010 subsample, IP growth h=12:')
print(f"  Adjusted R2, baseline:        {res_sub['baseline'].rsquared_adj:.3f}")
print(f"  Adjusted R2, + Baa-Aaa spread: {res_sub['with_credit_spread'].rsquared_adj:.3f}")
print(f"  Coefficient on Baa-Aaa spread: {res_sub['with_credit_spread'].params['baa_aaa']:.3f} "
      f"(t = {res_sub['with_credit_spread'].tvalues['baa_aaa']:.2f})")

## 10. Edge Cases: Does the Relationship Hold in Every Decade?

We re-estimate the same regression decade by decade. If the credit-spread coefficient is unstable in sign or magnitude across decades, that is an edge case the single full-sample regression conceals, and it should temper how much weight the full-sample point estimate deserves.

In [ ]:
decades = [('1973-01-01', '1982-12-31'), ('1983-01-01', '1992-12-31'),
           ('1993-01-01', '2002-12-31'), ('2003-01-01', '2012-12-31'),
           ('2013-01-01', '2026-12-31')]

rows = []
for start, end in decades:
    sub = df.loc[start:end]
    try:
        r = forecasting_regression(sub, 'ip_growth_h12', h=12)
        rows.append({
            'window': f'{start[:4]}-{end[:4]}',
            'n': r['n'],
            'coef_baa_aaa': r['with_credit_spread'].params.get('baa_aaa', np.nan),
            't_stat': r['with_credit_spread'].tvalues.get('baa_aaa', np.nan),
            'adj_r2_full': r['with_credit_spread'].rsquared_adj,
        })
    except Exception as e:
        rows.append({'window': f'{start[:4]}-{end[:4]}', 'n': len(sub), 'coef_baa_aaa': np.nan,
                      't_stat': np.nan, 'adj_r2_full': np.nan})

pd.DataFrame(rows)

**Interpretation.** A decade-by-decade breakdown is exactly the kind of check the published paper does not report (it reports only the full sample and a single 1985-2010 split), so this cell is a genuine extension of the paper's robustness section, not a reproduction of it. Pay particular attention to whether the coefficient flips sign or loses significance in any decade; the post-2013 window, in particular, has experienced two unusual episodes (the 2020 pandemic shock and the 2022-23 rate-hiking cycle) that are very different in character from the 2008 financial crisis that motivates the paper, and the Baa-Aaa spread may behave differently around a demand shock than around a financial-intermediation shock.

## 11. Contradictions: When Would the Result Fail?

The paper's own mechanism is that the credit spread signals a *reduction in the effective risk-bearing capacity of the financial sector*, not simply higher expected defaults. The Baa-Aaa spread, unlike the GZ spread, is not decomposed into an expected-default component and an excess bond premium, so it should, in principle, do a worse job isolating financial-sector stress from ordinary default-risk repricing. We test this directly by comparing the credit spread's forecasting power in two episodes with very different mechanisms: 2007-2009 (a genuine financial-intermediation crisis, exactly the episode the paper's excess bond premium was designed to explain) versus 2020 (a real, non-financial shock in which credit spreads widened briefly but the shock's origin was a pandemic-induced demand and supply collapse, not a deterioration in dealer balance sheets).

In [ ]:
for label, start, end in [('Global Financial Crisis window', '2005-01-01', '2011-12-31'),
                           ('Covid shock window', '2017-01-01', '2023-12-31')]:
    sub = df.loc[start:end]
    r = forecasting_regression(sub, 'ip_growth_h12', h=12)
    print(f"{label} ({start[:4]}-{end[:4]}, n={r['n']}):")
    print(f"  Coefficient on Baa-Aaa spread: {r['with_credit_spread'].params['baa_aaa']:.3f} "
          f"(t = {r['with_credit_spread'].tvalues['baa_aaa']:.2f})")
    print(f"  Adjusted R2: {r['with_credit_spread'].rsquared_adj:.3f}\n")

**Interpretation.** If the credit spread's coefficient is materially weaker (smaller in magnitude, less significant, or a worse fit) around the Covid window than around the financial-crisis window, that is evidence in favour of the paper's own theoretical distinction between default-risk shocks and financial-intermediation shocks: a demand shock that is not routed through impaired bank and dealer balance sheets should leave a smaller fingerprint on the excess-bond-premium-like component of credit spreads, even if headline spreads still widen mechanically because expected defaults rise. This is precisely the contradiction the paper's own theory predicts, so finding it here is a genuine (if indirect) validation of the paper's excess bond premium decomposition, obtained without needing the proprietary bond-level data.

## 12. Alternative Specifications

In [ ]:
d = sample_1973_2010.dropna(subset=['ip_growth_h12', 'term_spread', 'short_rate', 'ip_growth_l1', 'baa_aaa']).copy()

specs = {
    'AR(1) only': ['ip_growth_l1'],
    'Term spread only': ['ip_growth_l1', 'term_spread'],
    'Credit spread only': ['ip_growth_l1', 'baa_aaa'],
    'Term + credit spread': ['ip_growth_l1', 'term_spread', 'baa_aaa'],
    'Full (term + short rate + credit spread)': ['ip_growth_l1', 'term_spread', 'short_rate', 'baa_aaa'],
}

spec_rows = []
for name, cols in specs.items():
    X = sm.add_constant(d[cols])
    m = sm.OLS(d['ip_growth_h12'], X).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
    spec_rows.append({'specification': name, 'adj_r2': m.rsquared_adj,
                       'credit_spread_coef': m.params.get('baa_aaa', np.nan)})

pd.DataFrame(spec_rows)

## 13. Counterfactuals: What Would Missing the Credit-Spread Signal Have Cost?

In [ ]:
d2 = sample_1973_2010.dropna(subset=['ip_growth_h12', 'term_spread', 'short_rate', 'ip_growth_l1', 'baa_aaa']).copy()
X_base = sm.add_constant(d2[['ip_growth_l1', 'term_spread', 'short_rate']])
X_full = sm.add_constant(d2[['ip_growth_l1', 'term_spread', 'short_rate', 'baa_aaa']])
y = d2['ip_growth_h12']

m_base = sm.OLS(y, X_base).fit()
m_full = sm.OLS(y, X_full).fit()

d2['fitted_baseline'] = m_base.fittedvalues
d2['fitted_full'] = m_full.fittedvalues

window = d2.loc['2007-01-01':'2009-12-31']
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(window.index, window['ip_growth_h12'], color='#172A3A', lw=1.4, label='Actual 12-month IP growth')
ax.plot(window.index, window['fitted_baseline'], color='#C8AE82', lw=1.2, ls='--', label='Fitted, without credit spread')
ax.plot(window.index, window['fitted_full'], color='#3F4A32', lw=1.2, label='Fitted, with credit spread')
ax.axhline(0, color='black', lw=0.6)
ax.set_title('Counterfactual: Forecasting Industrial Production Growth Into the 2008-09 Crisis')
ax.set_ylabel('Annualised 12-month IP growth (pct)')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

rmse_base = np.sqrt(np.mean((window['ip_growth_h12'] - window['fitted_baseline'])**2))
rmse_full = np.sqrt(np.mean((window['ip_growth_h12'] - window['fitted_full'])**2))
print(f"In-sample RMSE, 2007-2009 window, without credit spread: {rmse_base:.2f}")
print(f"In-sample RMSE, 2007-2009 window, with credit spread:    {rmse_full:.2f}")

## 14. Extension: Pseudo Out-of-Sample Forecast Evaluation

The paper evaluates its regressions entirely in sample. A natural, genuine extension is to ask whether the credit spread would actually have improved *real-time* forecasts, using a rolling window that only uses information available up to the forecast origin. This is a harder, more realistic test than the in-sample R-squared comparisons in Table 2 of the paper, and it is not attempted there.

In [ ]:
def rolling_oos_rmse(data, y_col, feature_cols, min_train=120, h=12):
    d = data.dropna(subset=[y_col] + feature_cols).reset_index()
    errors = []
    for t in range(min_train, len(d) - h):
        train = d.iloc[:t]
        X_train = sm.add_constant(train[feature_cols])
        y_train = train[y_col]
        model = sm.OLS(y_train, X_train).fit()
        X_test = sm.add_constant(d.iloc[[t]][feature_cols], has_constant='add')
        pred = model.predict(X_test).iloc[0]
        actual = d.iloc[t][y_col]
        errors.append((pred - actual) ** 2)
    return np.sqrt(np.mean(errors)), len(errors)

base_cols = ['ip_growth_l1', 'term_spread', 'short_rate']
full_cols = ['ip_growth_l1', 'term_spread', 'short_rate', 'baa_aaa']

rmse_oos_base, n_base = rolling_oos_rmse(sample_1973_2010, 'ip_growth_h12', base_cols)
rmse_oos_full, n_full = rolling_oos_rmse(sample_1973_2010, 'ip_growth_h12', full_cols)

print(f"Pseudo out-of-sample RMSE, 12-month IP growth, expanding window, min 10 years training data:")
print(f"  Without credit spread: {rmse_oos_base:.3f}  (n forecasts = {n_base})")
print(f"  With credit spread:    {rmse_oos_full:.3f}  (n forecasts = {n_full})")
print(f"  Improvement: {100 * (1 - rmse_oos_full / rmse_oos_base):.1f} percent lower RMSE")

**Interpretation.** In-sample R-squared improvements, of the kind reported throughout the paper, can overstate real-world forecasting value because the same data are used to fit and evaluate the model. The expanding-window exercise above is a stricter test: at each point in time, the model only sees data that would actually have been available to a forecaster at that date. If the out-of-sample RMSE improvement is much smaller than the in-sample R-squared gain would suggest, that is a genuine, quantified caveat to the paper's practical forecasting claims, obtained through extension rather than criticism of the published results (the paper does not run this test, so this is not a like-for-like comparison, only a complementary one).

## 15. Summary and Interpretation

This notebook cannot and does not reproduce the paper's central contribution, the GZ credit spread and the excess bond premium, because those objects require proprietary bond-level pricing data. What it does demonstrate, using genuine public data from FRED, is:

1. A coarse, publicly available credit spread (Baa-Aaa) shows the same qualitative countercyclical pattern that motivates the paper (Section 8 above), rising sharply around every NBER-dated recession in the sample.
2. Adding this spread to a standard term-spread-and-short-rate forecasting regression raises the in-sample fit for future industrial production growth, in the same direction as, but by construction weaker than, the paper's own GZ spread results (Section 7), which is exactly what the paper's own comparison of the GZ spread against the Baa-Aaa spread would predict.
3. The forecasting relationship is not stable across decades or across the source of the underlying shock (Sections 10 and 11), which is consistent with, and lends indirect support to, the paper's theoretical distinction between ordinary default-risk repricing and shocks to the risk-bearing capacity of financial intermediaries.
4. Out-of-sample forecast gains are more modest than in-sample gains (Section 14), a caveat worth carrying into the main written review's discussion of what the paper does and does not establish.

Where this notebook's Baa-Aaa results diverge sharply from the paper's own reported GZ-spread coefficients (see the written review, Level 3 discussion), that divergence is itself evidence for the paper's central methodological claim: the bottom-up, micro-founded construction of the GZ spread is not a cosmetic refinement of a generic credit spread, it materially changes the signal.